
# Notebook 20 — Residual Universality Classes and Trajectory Renormalization

This notebook builds directly from Notebook 19.

Notebook 19 asked:

```text
How do topology classes move through residual manifold space as graph size increases?
```

Notebook 20 asks:

```text
Do different residual trajectories collapse into shared universality classes?
```

Core frame:

```text
Residual topology identity can be studied as a scale-dependent trajectory.
Shared trajectory geometry suggests shared residual universality structure.
```

This notebook is Colab-safe and self-contained:
- it loads Notebook 19 outputs if available,
- otherwise it rebuilds compatible residual phase trajectory data internally,
- it exports figures, CSVs, JSON, markdown notes, and an optional zip.


In [ ]:

import json
import zipfile
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

np.random.seed(42)

# ---------------------------------------------------
# robust repo detection for Colab and local Jupyter
# ---------------------------------------------------

def detect_repo_root():
    cwd = Path.cwd()

    if (cwd / "notebooks").exists() or (cwd / ".git").exists():
        return cwd

    if cwd.name == "notebooks":
        return cwd.parent

    search_root = Path("/content") if Path("/content").exists() else cwd

    for name in [
        "residual_phase_trajectory_embedding.csv",
        "residual_phase_trajectory_metrics.csv",
        "residual_geometry_features.csv",
        "residual_classification_feature_matrix.csv",
    ]:
        matches = list(search_root.rglob(name))
        if matches:
            return matches[0].parents[1]

    return Path("/content") if Path("/content").exists() else cwd

REPO_ROOT = detect_repo_root()
RESULTS_DIR = REPO_ROOT / "results"
FIG_DIR = REPO_ROOT / "figures"
DOCS_DIR = REPO_ROOT / "docs"

RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

GRAPH_SIZES = [16, 32, 64, 128]

TOPOLOGIES = [
    "ring_lattice",
    "small_world",
    "erdos_renyi",
    "scale_free",
    "modular_clustered",
]

TOPOLOGY_LABELS = {
    "ring_lattice": "ring lattice",
    "small_world": "small world",
    "erdos_renyi": "Erdős–Rényi",
    "scale_free": "scale free",
    "modular_clustered": "modular clustered",
}

print("cwd:", Path.cwd())
print("repo root:", REPO_ROOT)
print("results dir:", RESULTS_DIR)
print("figures dir:", FIG_DIR)
print("docs dir:", DOCS_DIR)


## 1. Load trajectory data or regenerate compatible inputs

In [ ]:

# ---------------------------------------------------
# fallback synthetic residual-geometry model
# consistent with Notebook 19's self-contained version
# ---------------------------------------------------

NOISE_GRID = np.linspace(0.0, 0.40, 81)
MIDPOINT_LEVEL = 0.50

def logistic_z(z):
    return 1 / (1 + np.exp(-z))

def shared_profile(z):
    return 1 - logistic_z(z)

TOPOLOGY_PARAMS = {
    "ring_lattice": {
        "eta_inf": 0.135, "eta_shift": 0.060,
        "sigma_inf": 0.030, "sigma_scale": 0.080,
        "nu": 0.45, "beta": 0.40,
        "fragment_strength": 0.08, "modifier": 0.98,
    },
    "small_world": {
        "eta_inf": 0.160, "eta_shift": 0.075,
        "sigma_inf": 0.038, "sigma_scale": 0.095,
        "nu": 0.48, "beta": 0.37,
        "fragment_strength": 0.06, "modifier": 1.02,
    },
    "erdos_renyi": {
        "eta_inf": 0.120, "eta_shift": 0.055,
        "sigma_inf": 0.028, "sigma_scale": 0.075,
        "nu": 0.42, "beta": 0.45,
        "fragment_strength": 0.10, "modifier": 0.96,
    },
    "scale_free": {
        "eta_inf": 0.105, "eta_shift": 0.052,
        "sigma_inf": 0.025, "sigma_scale": 0.070,
        "nu": 0.44, "beta": 0.50,
        "fragment_strength": 0.14, "modifier": 0.93,
    },
    "modular_clustered": {
        "eta_inf": 0.092, "eta_shift": 0.048,
        "sigma_inf": 0.023, "sigma_scale": 0.065,
        "nu": 0.40, "beta": 0.52,
        "fragment_strength": 0.18, "modifier": 0.90,
    },
}

def finite_size_eta_c(params, N):
    return params["eta_inf"] + params["eta_shift"] * (N ** (-params["nu"]))

def finite_size_sigma(params, N):
    return params["sigma_inf"] + params["sigma_scale"] * (N ** (-params["beta"]))

def simulate_cgcs_curve(topology, N, noise_grid, repeat=0):
    p = TOPOLOGY_PARAMS[topology]
    eta_c = finite_size_eta_c(p, N)
    sigma = finite_size_sigma(p, N)

    z = (noise_grid - eta_c) / sigma
    base = shared_profile(z)

    central_weight = np.exp(-0.5 * z**2)
    outside_weight = 1 - central_weight
    fragment = (
        p["fragment_strength"]
        * outside_weight
        * logistic_z((noise_grid - eta_c) / (2.0 * sigma))
    )

    rng = np.random.default_rng(
        40_000 + repeat + N + sum(ord(c) for c in topology)
    )
    noise_term = rng.normal(0, 0.010 * np.sqrt(32 / N), size=len(noise_grid))

    return np.clip(p["modifier"] * base - fragment + noise_term, 0, 1)

def extract_transition_metrics(noise, cgcs):
    noise = np.asarray(noise, dtype=float)
    cgcs = np.asarray(cgcs, dtype=float)

    midpoint_idx = int(np.argmin(np.abs(cgcs - MIDPOINT_LEVEL)))
    eta_mid = float(noise[midpoint_idx])

    deriv = np.gradient(cgcs, noise)
    max_abs_slope = float(np.max(np.abs(deriv)))
    sigma_est = float(1 / max(4 * max_abs_slope, 1e-6))

    return eta_mid, sigma_est

def normalized_entropy_from_energy(z, energy, bins=24):
    hist, _ = np.histogram(z, bins=bins, range=(-6, 6), weights=energy)
    total = hist.sum()
    if total <= 0:
        return 0.0
    p = hist / total
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)) / np.log(bins))

def regenerate_geometry_features():
    rows = []
    repeats = 24

    for N in GRAPH_SIZES:
        for topology in TOPOLOGIES:
            curves = []
            for repeat in range(repeats):
                curves.append(simulate_cgcs_curve(topology, N, NOISE_GRID, repeat))

            mean_curve = np.array(curves).mean(axis=0)
            eta_mid, sigma_est = extract_transition_metrics(NOISE_GRID, mean_curve)
            sigma_est = max(sigma_est, 1e-6)

            z = (NOISE_GRID - eta_mid) / sigma_est
            predicted = shared_profile(z)
            residual = mean_curve - predicted

            energy = residual ** 2
            abs_res = np.abs(residual)

            left_energy = float(np.sum(energy[z < 0]))
            right_energy = float(np.sum(energy[z >= 0]))
            lr_total = left_energy + right_energy

            k = max(1, int(np.ceil(0.10 * len(energy))))
            localization = float(np.sort(energy)[-k:].sum() / max(np.sum(energy), 1e-12))

            # simple bend energy on fixed grid
            order = np.argsort(z)
            z_order = z[order]
            r_order = residual[order]
            z_grid = np.linspace(-6, 6, 241)
            r_grid = np.interp(z_grid, z_order, r_order, left=np.nan, right=np.nan)
            valid = np.isfinite(r_grid)
            if valid.sum() > 5:
                rv = r_grid[valid]
                zv = z_grid[valid]
                first = np.gradient(rv, zv)
                second = np.gradient(first, zv)
                bend_energy = float(np.trapz(second**2, zv))
                mean_abs_bend = float(np.mean(np.abs(second)))
            else:
                bend_energy = 0.0
                mean_abs_bend = 0.0

            # low/high residual spectral ratio
            fill = np.nanmean(r_grid)
            r_fft = np.where(np.isfinite(r_grid), r_grid, fill)
            r_fft = r_fft - np.mean(r_fft)
            power = np.abs(np.fft.rfft(r_fft)) ** 2
            power[0] = 0
            total_power = float(power.sum())
            if total_power > 0:
                cutoff = max(2, int(0.20 * len(power)))
                low = float(power[1:cutoff].sum() / total_power)
                high = float(power[cutoff:].sum() / total_power)
                spectral_ratio = float(high / max(low, 1e-9))
            else:
                spectral_ratio = 0.0

            rows.append({
                "topology": topology,
                "label": TOPOLOGY_LABELS[topology],
                "n_modules": int(N),
                "mean_abs_residual": float(np.mean(abs_res)),
                "max_abs_residual": float(np.max(abs_res)),
                "total_residual_energy": float(np.sum(energy)),
                "residual_localization": localization,
                "residual_asymmetry": float((right_energy - left_energy) / lr_total) if lr_total > 0 else 0.0,
                "residual_entropy": normalized_entropy_from_energy(z, energy),
                "residual_bend_energy": bend_energy,
                "mean_abs_bend": mean_abs_bend,
                "residual_spectral_ratio": spectral_ratio,
            })

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_DIR / "residual_geometry_features.csv", index=False)
    return df

def build_embedding_from_features(feature_df):
    candidate_cols = [
        "mean_abs_residual",
        "max_abs_residual",
        "total_residual_energy",
        "residual_localization",
        "residual_asymmetry",
        "residual_entropy",
        "residual_bend_energy",
        "mean_abs_bend",
        "residual_spectral_ratio",
    ]
    feature_cols = [c for c in candidate_cols if c in feature_df.columns]
    X = feature_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)

    Xs = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2)
    Z = pca.fit_transform(Xs)

    embedding = feature_df[["topology", "label", "n_modules"]].copy()
    embedding["pc1"] = Z[:, 0]
    embedding["pc2"] = Z[:, 1]
    embedding.to_csv(RESULTS_DIR / "residual_phase_trajectory_embedding.csv", index=False)

    return embedding, pca.explained_variance_ratio_, feature_cols

embedding_path = RESULTS_DIR / "residual_phase_trajectory_embedding.csv"
feature_path = RESULTS_DIR / "residual_geometry_features.csv"

if embedding_path.exists():
    traj_df = pd.read_csv(embedding_path)
    data_source = "loaded Notebook 19 residual phase trajectory embedding"
    pca_variance = [np.nan, np.nan]
    feature_cols = []
elif feature_path.exists():
    feature_df = pd.read_csv(feature_path)
    traj_df, pca_variance, feature_cols = build_embedding_from_features(feature_df)
    data_source = "rebuilt trajectory embedding from residual geometry features"
else:
    feature_df = regenerate_geometry_features()
    traj_df, pca_variance, feature_cols = build_embedding_from_features(feature_df)
    data_source = "regenerated residual geometry features and trajectory embedding internally"

traj_df["label"] = traj_df["topology"].map(TOPOLOGY_LABELS).fillna(traj_df.get("label", traj_df["topology"]))
traj_df = traj_df.sort_values(["topology", "n_modules"]).reset_index(drop=True)

print("data source:", data_source)
print("trajectory rows:", traj_df.shape)
traj_df.head()


## 2. Trajectory reconstruction

In [ ]:

plt.figure(figsize=(10, 8))

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    if sub.empty:
        continue

    plt.plot(
        sub["pc1"],
        sub["pc2"],
        marker="o",
        linewidth=2,
        markersize=8,
        label=TOPOLOGY_LABELS[topology],
    )

    for i in range(len(sub) - 1):
        x0, y0 = sub.iloc[i][["pc1", "pc2"]]
        x1, y1 = sub.iloc[i + 1][["pc1", "pc2"]]
        plt.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(arrowstyle="->", lw=1.2, alpha=0.75),
        )

    for _, row in sub.iterrows():
        plt.annotate(
            f"N={int(row['n_modules'])}",
            xy=(row["pc1"], row["pc2"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
            alpha=0.8,
        )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)

xlabel = "PC1" if np.isnan(pca_variance[0]) else f"PC1 ({pca_variance[0]:.1%})"
ylabel = "PC2" if np.isnan(pca_variance[1]) else f"PC2 ({pca_variance[1]:.1%})"

plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.title("Residual universality trajectory reconstruction")
plt.grid(alpha=0.3)
plt.legend(fontsize=9)

fig_path = FIG_DIR / "universality_trajectory_reconstruction.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## 3. Flow descriptors

In [ ]:

def trajectory_points(topology):
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    return sub[["pc1", "pc2"]].to_numpy(dtype=float), sub["n_modules"].to_numpy(dtype=int)

def path_length(points):
    if len(points) < 2:
        return 0.0
    return float(np.sum(np.linalg.norm(np.diff(points, axis=0), axis=1)))

def endpoint_distance(points):
    if len(points) < 2:
        return 0.0
    return float(np.linalg.norm(points[-1] - points[0]))

def curvature_angle_sum(points):
    if len(points) < 3:
        return 0.0

    total = 0.0
    for i in range(1, len(points) - 1):
        a = points[i] - points[i - 1]
        b = points[i + 1] - points[i]
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        if denom > 0:
            c = float(np.clip(np.dot(a, b) / denom, -1, 1))
            total += math.acos(c)
    return float(total)

def directional_entropy(points, bins=8):
    if len(points) < 2:
        return 0.0

    diffs = np.diff(points, axis=0)
    angles = np.arctan2(diffs[:, 1], diffs[:, 0])
    hist, _ = np.histogram(angles, bins=bins, range=(-np.pi, np.pi))
    total = hist.sum()
    if total <= 0:
        return 0.0

    p = hist / total
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)) / np.log(bins))

def early_late_persistence(points):
    if len(points) < 4:
        return np.nan

    early = points[1] - points[0]
    late = points[-1] - points[-2]
    denom = np.linalg.norm(early) * np.linalg.norm(late)
    if denom <= 0:
        return np.nan
    return float(np.dot(early, late) / denom)

descriptor_rows = []

for topology in TOPOLOGIES:
    points, sizes = trajectory_points(topology)
    if len(points) == 0:
        continue

    L = path_length(points)
    D = endpoint_distance(points)
    K = curvature_angle_sum(points)
    H = directional_entropy(points)
    P = early_late_persistence(points)
    tortuosity = float(L / max(D, 1e-9))

    descriptor_rows.append({
        "topology": topology,
        "label": TOPOLOGY_LABELS[topology],
        "n_points": int(len(points)),
        "trajectory_length": L,
        "endpoint_distance": D,
        "turning_angle_sum": K,
        "directional_entropy": H,
        "scale_persistence": P,
        "trajectory_tortuosity": tortuosity,
    })

flow_df = pd.DataFrame(descriptor_rows)
flow_df.to_csv(RESULTS_DIR / "universality_flow_descriptors.csv", index=False)

flow_df


In [ ]:

metrics = [
    "trajectory_length",
    "endpoint_distance",
    "turning_angle_sum",
    "directional_entropy",
    "trajectory_tortuosity",
]

fig, axes = plt.subplots(1, len(metrics), figsize=(22, 5))

for ax, metric in zip(axes, metrics):
    plot_df = flow_df.sort_values(metric, ascending=True)
    ax.barh(plot_df["label"], plot_df[metric])
    ax.set_title(metric.replace("_", " "))
    ax.grid(alpha=0.3, axis="x")

plt.suptitle("Residual universality flow descriptors", fontsize=16)
plt.tight_layout()

fig_path = FIG_DIR / "universality_flow_descriptors.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## 4. Universality similarity matrix

In [ ]:

def resample_path(points, n=32):
    points = np.asarray(points, dtype=float)
    if len(points) == 1:
        return np.repeat(points, n, axis=0)

    seg = np.linalg.norm(np.diff(points, axis=0), axis=1)
    arc = np.concatenate([[0], np.cumsum(seg)])

    if arc[-1] <= 0:
        return np.repeat(points[:1], n, axis=0)

    target = np.linspace(0, arc[-1], n)
    out = np.zeros((n, 2))
    out[:, 0] = np.interp(target, arc, points[:, 0])
    out[:, 1] = np.interp(target, arc, points[:, 1])
    return out

def flattened_direction_vector(points):
    resampled = resample_path(points, n=32)
    diffs = np.diff(resampled, axis=0).reshape(-1)
    norm = np.linalg.norm(diffs)
    if norm <= 0:
        return diffs
    return diffs / norm

def discrete_frechet(P, Q):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)
    ca = np.full((len(P), len(Q)), -1.0)

    def c(i, j):
        if ca[i, j] > -1:
            return ca[i, j]
        dist = np.linalg.norm(P[i] - Q[j])

        if i == 0 and j == 0:
            ca[i, j] = dist
        elif i > 0 and j == 0:
            ca[i, j] = max(c(i - 1, 0), dist)
        elif i == 0 and j > 0:
            ca[i, j] = max(c(0, j - 1), dist)
        elif i > 0 and j > 0:
            ca[i, j] = max(
                min(c(i - 1, j), c(i - 1, j - 1), c(i, j - 1)),
                dist,
            )
        else:
            ca[i, j] = float("inf")
        return ca[i, j]

    return float(c(len(P) - 1, len(Q) - 1))

def dtw_distance(P, Q):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)
    D = np.full((len(P) + 1, len(Q) + 1), np.inf)
    D[0, 0] = 0.0

    for i in range(1, len(P) + 1):
        for j in range(1, len(Q) + 1):
            cost = np.linalg.norm(P[i - 1] - Q[j - 1])
            D[i, j] = cost + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])

    return float(D[len(P), len(Q)] / (len(P) + len(Q)))

paths = {}
directions = {}

for topology in TOPOLOGIES:
    points, _ = trajectory_points(topology)
    if len(points) > 0:
        paths[topology] = points
        directions[topology] = flattened_direction_vector(points)

available = list(paths.keys())
n = len(available)

cosine_alignment = np.zeros((n, n))
frechet = np.zeros((n, n))
dtw = np.zeros((n, n))

for i, a in enumerate(available):
    for j, b in enumerate(available):
        u = directions[a]
        v = directions[b]
        denom = np.linalg.norm(u) * np.linalg.norm(v)
        cosine_alignment[i, j] = float(np.dot(u, v) / denom) if denom > 0 else 0.0

        Pa = resample_path(paths[a], n=32)
        Pb = resample_path(paths[b], n=32)
        frechet[i, j] = discrete_frechet(Pa, Pb)
        dtw[i, j] = dtw_distance(Pa, Pb)

def normalize_distance_matrix(M):
    M = np.asarray(M, dtype=float)
    offdiag = M[~np.eye(M.shape[0], dtype=bool)]
    scale = np.nanmax(offdiag) if len(offdiag) else 1.0
    if scale <= 0 or not np.isfinite(scale):
        scale = 1.0
    return M / scale

frechet_norm = normalize_distance_matrix(frechet)
dtw_norm = normalize_distance_matrix(dtw)

# similarity combines alignment and inverse distances
similarity = (
    0.50 * ((cosine_alignment + 1) / 2)
    + 0.25 * (1 - frechet_norm)
    + 0.25 * (1 - dtw_norm)
)
similarity = np.clip(similarity, 0, 1)
np.fill_diagonal(similarity, 1.0)

similarity_df = pd.DataFrame(
    similarity,
    index=[TOPOLOGY_LABELS[t] for t in available],
    columns=[TOPOLOGY_LABELS[t] for t in available],
)

similarity_out = similarity_df.copy()
similarity_out.insert(0, "topology", similarity_out.index)
similarity_out.to_csv(RESULTS_DIR / "universality_similarity_matrix.csv", index=False)

plt.figure(figsize=(8, 7))
im = plt.imshow(similarity, vmin=0, vmax=1)

plt.xticks(range(n), [TOPOLOGY_LABELS[t] for t in available], rotation=45, ha="right")
plt.yticks(range(n), [TOPOLOGY_LABELS[t] for t in available])

for i in range(n):
    for j in range(n):
        plt.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center")

plt.title("Residual universality similarity")
plt.colorbar(im, fraction=0.046, pad=0.04, label="combined similarity")
plt.tight_layout()

fig_path = FIG_DIR / "universality_similarity_heatmap.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

similarity_df


## 5. Residual renormalization flow graph

In [ ]:

# Draw a simple directed flow graph without networkx dependency.
# Nodes are topology states at N=16,32,64,128.
# Edges connect each topology's scale progression.

plt.figure(figsize=(12, 8))

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    if sub.empty:
        continue

    xs = sub["pc1"].to_numpy()
    ys = sub["pc2"].to_numpy()

    plt.plot(xs, ys, marker="o", linewidth=2, label=TOPOLOGY_LABELS[topology])

    for i in range(len(sub) - 1):
        x0, y0 = xs[i], ys[i]
        x1, y1 = xs[i + 1], ys[i + 1]
        dx, dy = x1 - x0, y1 - y0
        edge_len = math.sqrt(dx * dx + dy * dy)
        alpha = min(1.0, 0.35 + 0.20 * edge_len)

        plt.arrow(
            x0,
            y0,
            dx * 0.85,
            dy * 0.85,
            head_width=0.08,
            length_includes_head=True,
            alpha=alpha,
        )

    for _, row in sub.iterrows():
        plt.text(
            row["pc1"],
            row["pc2"],
            f" {TOPOLOGY_LABELS[topology]}\n N={int(row['n_modules'])}",
            fontsize=8,
            alpha=0.85,
        )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.xlabel("residual manifold coordinate 1")
plt.ylabel("residual manifold coordinate 2")
plt.title("Residual renormalization flow graph")
plt.grid(alpha=0.3)
plt.legend(fontsize=9, loc="best")

fig_path = FIG_DIR / "residual_renormalization_flow_graph.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## 6. Universality clustering

In [ ]:

distance = 1 - similarity
np.fill_diagonal(distance, 0.0)

# squareform requires symmetry and zero diagonal
distance = 0.5 * (distance + distance.T)
np.fill_diagonal(distance, 0.0)

condensed = squareform(distance)
Z_link = linkage(condensed, method="average")

plt.figure(figsize=(9, 5))
dendrogram(
    Z_link,
    labels=[TOPOLOGY_LABELS[t] for t in available],
    leaf_rotation=30,
)
plt.ylabel("1 - universality similarity")
plt.title("Residual universality dendrogram")
plt.tight_layout()

fig_path = FIG_DIR / "universality_dendrogram.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

cluster_labels = fcluster(Z_link, t=0.45, criterion="distance")

cluster_df = pd.DataFrame({
    "topology": available,
    "label": [TOPOLOGY_LABELS[t] for t in available],
    "universality_cluster": cluster_labels.astype(int),
})
cluster_df.to_csv(RESULTS_DIR / "universality_clusters.csv", index=False)

cluster_df


## 7. Scale transition stability

In [ ]:

transition_rows = []

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    points = sub[["pc1", "pc2"]].to_numpy(dtype=float)
    sizes = sub["n_modules"].to_numpy(dtype=int)

    for i in range(len(points) - 1):
        step = points[i + 1] - points[i]
        transition_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "from_N": int(sizes[i]),
            "to_N": int(sizes[i + 1]),
            "transition": f"{int(sizes[i])}→{int(sizes[i + 1])}",
            "dx": float(step[0]),
            "dy": float(step[1]),
            "step_magnitude": float(np.linalg.norm(step)),
            "angle": float(np.arctan2(step[1], step[0])),
        })

transition_df = pd.DataFrame(transition_rows)

# local directional reversals between consecutive scale steps
reversal_rows = []

for topology in TOPOLOGIES:
    sub = transition_df[transition_df["topology"] == topology].sort_values("from_N")
    vecs = sub[["dx", "dy"]].to_numpy(dtype=float)

    for i in range(len(vecs) - 1):
        u = vecs[i]
        v = vecs[i + 1]
        denom = np.linalg.norm(u) * np.linalg.norm(v)
        alignment = float(np.dot(u, v) / denom) if denom > 0 else np.nan

        reversal_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "transition_pair": f"{sub.iloc[i]['transition']} / {sub.iloc[i + 1]['transition']}",
            "local_alignment": alignment,
            "local_instability": float(1 - alignment) if np.isfinite(alignment) else np.nan,
        })

reversal_df = pd.DataFrame(reversal_rows)

transition_df.to_csv(RESULTS_DIR / "universality_transition_stability.csv", index=False)
reversal_df.to_csv(RESULTS_DIR / "universality_transition_reversals.csv", index=False)

pivot = transition_df.pivot_table(
    index="label",
    columns="transition",
    values="step_magnitude",
    aggfunc="mean",
)

plt.figure(figsize=(8, 6))
im = plt.imshow(pivot.to_numpy(), aspect="auto")

plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.to_numpy()[i, j]
        plt.text(j, i, f"{val:.2f}", ha="center", va="center")

plt.title("Scale transition instability")
plt.colorbar(im, fraction=0.046, pad=0.04, label="step magnitude")
plt.tight_layout()

fig_path = FIG_DIR / "scale_transition_instability.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

transition_df.head()


## 8. Universality manifold embedding

In [ ]:

descriptor_for_embedding = flow_df.merge(
    cluster_df[["topology", "universality_cluster"]],
    on="topology",
    how="left",
)

embed_cols = [
    "trajectory_length",
    "endpoint_distance",
    "turning_angle_sum",
    "directional_entropy",
    "trajectory_tortuosity",
]

X_flow = descriptor_for_embedding[embed_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)

if len(descriptor_for_embedding) >= 2:
    X_flow_scaled = StandardScaler().fit_transform(X_flow)
    pca_flow = PCA(n_components=2)
    U = pca_flow.fit_transform(X_flow_scaled)
else:
    U = np.zeros((len(descriptor_for_embedding), 2))
    pca_flow = None

descriptor_for_embedding["u1"] = U[:, 0]
descriptor_for_embedding["u2"] = U[:, 1]

descriptor_for_embedding.to_csv(RESULTS_DIR / "universality_manifold_embedding.csv", index=False)

plt.figure(figsize=(8, 6))

for _, row in descriptor_for_embedding.iterrows():
    plt.scatter(row["u1"], row["u2"], s=180, alpha=0.8)
    plt.annotate(
        row["label"],
        xy=(row["u1"], row["u2"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=10,
    )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)

if pca_flow is not None:
    xlabel = f"universality PC1 ({pca_flow.explained_variance_ratio_[0]:.1%})"
    ylabel = f"universality PC2 ({pca_flow.explained_variance_ratio_[1]:.1%})"
else:
    xlabel = "universality coordinate 1"
    ylabel = "universality coordinate 2"

plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.title("Universality manifold embedding")
plt.grid(alpha=0.3)

fig_path = FIG_DIR / "universality_manifold_embedding.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

descriptor_for_embedding


## 9. Summary table

In [ ]:

summary_df = flow_df.merge(
    cluster_df[["topology", "universality_cluster"]],
    on="topology",
    how="left",
)

# add mean transition magnitude
mean_step = (
    transition_df.groupby("topology")["step_magnitude"]
    .mean()
    .reset_index()
    .rename(columns={"step_magnitude": "mean_scale_step"})
)

summary_df = summary_df.merge(mean_step, on="topology", how="left")
summary_df = summary_df.sort_values(["universality_cluster", "trajectory_length"]).reset_index(drop=True)

summary_df.to_csv(RESULTS_DIR / "universality_summary.csv", index=False)
summary_df


## 10. Interpretation export

In [ ]:

# Select compact observations from computed values.

closest_pairs = []
for i in range(n):
    for j in range(i + 1, n):
        closest_pairs.append({
            "a": available[i],
            "b": available[j],
            "a_label": TOPOLOGY_LABELS[available[i]],
            "b_label": TOPOLOGY_LABELS[available[j]],
            "similarity": float(similarity[i, j]),
        })

closest_pairs = sorted(closest_pairs, key=lambda x: x["similarity"], reverse=True)

most_similar = closest_pairs[0] if closest_pairs else None
least_similar = closest_pairs[-1] if closest_pairs else None

max_length = summary_df.sort_values("trajectory_length", ascending=False).iloc[0]
max_curvature = summary_df.sort_values("turning_angle_sum", ascending=False).iloc[0]

interpretation = {
    "notebook": "20_residual_universality_classes.ipynb",
    "core_question": "Do different residual trajectories collapse into shared universality classes?",
    "core_claim": (
        "Residual topology families can be compared as scale-dependent trajectories. "
        "Shared trajectory similarity suggests residual universality classes."
    ),
    "data_source": data_source,
    "most_similar_pair": most_similar,
    "least_similar_pair": least_similar,
    "largest_trajectory_length": {
        "topology": str(max_length["topology"]),
        "label": str(max_length["label"]),
        "value": float(max_length["trajectory_length"]),
    },
    "largest_turning_angle_sum": {
        "topology": str(max_curvature["topology"]),
        "label": str(max_curvature["label"]),
        "value": float(max_curvature["turning_angle_sum"]),
    },
    "figures": [
        "universality_trajectory_reconstruction.png",
        "universality_flow_descriptors.png",
        "universality_similarity_heatmap.png",
        "residual_renormalization_flow_graph.png",
        "universality_dendrogram.png",
        "scale_transition_instability.png",
        "universality_manifold_embedding.png",
    ],
    "results": [
        "universality_flow_descriptors.csv",
        "universality_similarity_matrix.csv",
        "universality_transition_stability.csv",
        "universality_transition_reversals.csv",
        "universality_clusters.csv",
        "universality_manifold_embedding.csv",
        "universality_summary.csv",
        "universality_interpretation.json",
    ],
}

json_path = RESULTS_DIR / "universality_interpretation.json"
json_path.write_text(json.dumps(interpretation, indent=2), encoding="utf-8")

md = [
    "# Notebook 20 — Residual Universality Classes",
    "",
    "## Core question",
    "",
    "Do different residual trajectories collapse into shared universality classes?",
    "",
    "## Core claim",
    "",
    "Residual topology families can be compared as scale-dependent trajectories. Shared trajectory similarity suggests residual universality classes.",
    "",
    "## Recommended figures",
    "",
    "- `figures/universality_trajectory_reconstruction.png`",
    "- `figures/universality_similarity_heatmap.png`",
    "- `figures/universality_dendrogram.png`",
    "- `figures/universality_manifold_embedding.png`",
    "",
    "## Key computed observations",
    "",
]

if most_similar is not None:
    md.append(
        f"- Most similar pair: `{most_similar['a_label']}` and `{most_similar['b_label']}` "
        f"(similarity ≈ {most_similar['similarity']:.3f})."
    )

if least_similar is not None:
    md.append(
        f"- Least similar pair: `{least_similar['a_label']}` and `{least_similar['b_label']}` "
        f"(similarity ≈ {least_similar['similarity']:.3f})."
    )

md.append(
    f"- Largest trajectory length: `{max_length['label']}` "
    f"(length ≈ {max_length['trajectory_length']:.3f})."
)
md.append(
    f"- Largest trajectory curvature: `{max_curvature['label']}` "
    f"(turning angle ≈ {max_curvature['turning_angle_sum']:.3f})."
)

md_path = DOCS_DIR / "notebook_20_residual_universality_classes.md"
md_path.write_text("\n".join(md), encoding="utf-8")

print(json.dumps(interpretation, indent=2))
print("saved:", json_path)
print("saved:", md_path)


## Optional export zip

In [ ]:

zip_path = REPO_ROOT / "notebook_20_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file():
                    zf.write(file)

print(f"Created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))
